In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path(".../src").resolve()))

import os
import shutil

from manifest import manifestFaceAppFFHQ

ARREL_DFFD = Path("/home/jms/DFFD/ParentDir") 
ARREL_SORTIDA_FFD = Path("/home/jms/FFD_CVPR2020/data_faceapp_ffhq")  

LLAVOR = 42
NOM_VAL = "validation"  
USA_SYMLINKS = True

def crea_estructura_ffd(arrel_sortida: Path):
    for split in ["train", "eval", "test"]:
        for sub in ["Real", "Fake", "Mask"]:
            carpeta = arrel_sortida / split / sub
            if carpeta.exists():
                shutil.rmtree(carpeta)
            carpeta.mkdir(parents=True, exist_ok=True)


def crea_enllac_o_copia(origen: Path, desti: Path, usa_symlinks=True):
    if desti.exists() or desti.is_symlink():
        desti.unlink()

    if usa_symlinks:
        os.symlink(origen.resolve(), desti)
    else:
        shutil.copy2(origen, desti)


def obte_ruta_mask_faceapp(ruta_fake: Path) -> Path:

    split = ruta_fake.parent.name
    carpeta_faceapp = ruta_fake.parent.parent
    ruta_mask = carpeta_faceapp / f"{split}_mask" / ruta_fake.name

    if not ruta_mask.exists():
        raise FileNotFoundError(
            f"No s'ha trobat sa mascara per a {ruta_fake}\nEsperada a: {ruta_mask}"
        )

    return ruta_mask


def guarda_manifests(df_train, df_val, df_test, arrel_sortida: Path):
    carpeta_manifests = arrel_sortida / "manifests"
    carpeta_manifests.mkdir(parents=True, exist_ok=True)

    df_train.to_csv(carpeta_manifests / "train.csv", index=False)
    df_val.to_csv(carpeta_manifests / "val.csv", index=False)
    df_test.to_csv(carpeta_manifests / "test.csv", index=False)


def prepara_split_ffd(df, split_dest: str, arrel_sortida: Path, usa_symlinks=True):
    compt_real = 0
    compt_fake = 0

    for _, fila in df.iterrows():
        ruta_img = Path(fila["path"])

        if int(fila["label"]) == 0:
            nom_sortida = f"real_{compt_real:06d}{ruta_img.suffix.lower()}"
            desti_real = arrel_sortida / split_dest / "Real" / nom_sortida
            crea_enllac_o_copia(ruta_img, desti_real, usa_symlinks=usa_symlinks)
            compt_real += 1

        else:
            ruta_mask = obte_ruta_mask_faceapp(ruta_img)

            nom_sortida = f"fake_{compt_fake:06d}{ruta_img.suffix.lower()}"
            desti_fake = arrel_sortida / split_dest / "Fake" / nom_sortida
            desti_mask = arrel_sortida / split_dest / "Mask" / nom_sortida

            crea_enllac_o_copia(ruta_img, desti_fake, usa_symlinks=usa_symlinks)
            crea_enllac_o_copia(ruta_mask, desti_mask, usa_symlinks=usa_symlinks)

            compt_fake += 1


def main():
    df_train, df_val, df_test = manifestFaceAppFFHQ(
        arrel=ARREL_DFFD,
        llavor=LLAVOR
    )

    # guardam els manifests per usar exactament els mateixos a Phase Spectrum
    guarda_manifests(df_train, df_val, df_test, ARREL_SORTIDA_FFD)

    # cream estructura per a FFD
    crea_estructura_ffd(ARREL_SORTIDA_FFD)

    prepara_split_ffd(df_train, "train", ARREL_SORTIDA_FFD, usa_symlinks=USA_SYMLINKS)
    prepara_split_ffd(df_val, "eval", ARREL_SORTIDA_FFD, usa_symlinks=USA_SYMLINKS)
    prepara_split_ffd(df_test, "test", ARREL_SORTIDA_FFD, usa_symlinks=USA_SYMLINKS)

    print(f"Sortida: {ARREL_SORTIDA_FFD}")


if __name__ == "__main__":
    main()